# Poisson: learning the Green's function in 1-D

Let $K: f \mapsto u$ solve $-u'' = f$ on $(0,1)$ with $u(0) = u(1) = 0$. The functions $\phi_n(x) = \sqrt 2 \sin(n\pi x)$ are an orthonormal eigenbasis of $-\partial_{xx}$ with eigenvalues $\lambda_n = n^2\pi^2$, so $u_n = f_n / \lambda_n$ and

$$u(x) = \int_0^1 G(x,y) f(y)\,dy, \qquad G(x,y) = \sum_n \frac{\phi_n(x)\phi_n(y)}{\lambda_n} = \min\{x,y\} - xy.$$

A linear operator learned in the span of $\phi_m \otimes \phi_n$ is an integral operator with kernel $\tilde G(x,y) = \sum_{m,n} \langle \tilde K \phi_n, \phi_m\rangle\, \phi_m(x)\phi_n(y)$. We compare $\tilde G$ to $G$ as $N_{\mathrm{eff}}$ grows.

* Inputs/outputs: the first 208 sine coefficients (about 99.5% of the input energy).
* Input measure: $f_n \sim \mathrm{Jacobi}(n^2, n^2)$.
* Approximation space: linear operators on the first $N_{\mathrm{eff}}$ input modes, trained on `owls.sample_size(N_eff)` optimally sampled inputs, tested on 500 inputs $f \sim \rho$.
* The relative $L^2$ kernel error is computed exactly (Parseval) against the true $G$, including the modes beyond 208, which the learned kernel cannot represent.

This runs in under a minute. **Status:** rerun after the cleanup. In arXiv v1 the kernel used only the diagonal of the learned operator, and the plotted kernel error was squared. See `ERRATA.md`.

In [ ]:
QUICK = False      # True: tiny version of the experiment, written to a temporary folder (for testing)

import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import owls
from owls.plotting import paper_style, savefig

OUT = tempfile.mkdtemp() if QUICK else "."
DATA, FIGS = os.path.join(OUT, "data"), os.path.join(OUT, "figures")
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

In [ ]:
# Parameters
MAX_MODE = 208                     # number of sine modes for inputs and outputs
N_EFFS = [1] + list(range(10, 110, 10))
PLOT_N_EFFS = [1, 10, 100]         # pointwise errors shown in the figure
M_TEST = 500
GRID = 500                         # grid points per direction for the kernel plots
SEED = 0
if QUICK:
    MAX_MODE, N_EFFS, PLOT_N_EFFS, M_TEST, GRID = 30, [1, 5, 10], [1, 5, 10], 50, 50

modes = np.arange(1, MAX_MODE + 1)
measures = [owls.Jacobi(n**2) for n in modes]   # f_n ~ Jacobi(n^2, n^2), Var(f_n) = 1/(2n^2 + 3)
eigs = 1 / (np.pi**2 * modes**2)                # u_n = f_n / (n pi)^2

# Fraction of the input energy sum_n 1/(2n^2 + 3) carried by the first MAX_MODE modes
# (exact total: sum_n 1/(n^2 + a^2) = (pi a coth(pi a) - 1) / (2 a^2), with a^2 = 3/2)
a = np.sqrt(1.5)
total = 0.5 * (np.pi * a / np.tanh(np.pi * a) - 1) / (2 * a**2)
print(f"{MAX_MODE} modes carry {np.sum(1 / (2 * modes**2.0 + 3)) / total:.3%} of the input energy")

In [ ]:
x = np.linspace(0, 1, GRID)
P = np.sqrt(2) * np.sin(np.pi * np.outer(x, modes))   # P[i, n] = phi_n(x_i)
G_true = np.minimum.outer(x, x) - np.outer(x, x)      # G(x, y) = min(x, y) - xy   (symmetric)
G_norm2 = 1 / 90                                      # ||G||^2 = sum_n 1 / (n pi)^4 = 1/90

rel_boch, rel_kernel, Zdiffs = [], [], []
for i, n in enumerate(N_EFFS):
    rng = np.random.default_rng([SEED, i])
    L = owls.linear_index_set(n, MAX_MODE)
    model = owls.learn_operator(lambda f: f * eigs, measures, L, optimal=True, rng=rng)
    x_test = owls.sample_rho(M_TEST, measures, rng)
    rel_boch.append(owls.relative_bochner_error(model.predict(x_test), x_test * eigs))

    # Learned kernel: G~(x, y) = sum_{m, j} K[m, j] phi_m(x) phi_j(y), with K[m, j] = < K~ phi_j, phi_m >
    K = model.as_matrix()
    # Relative L^2 error: Frobenius error of the coefficients, plus the modes beyond MAX_MODE
    err2 = np.sum((K - np.diag(eigs)) ** 2) + (G_norm2 - np.sum(eigs**2))
    rel_kernel.append(np.sqrt(err2 / G_norm2))
    if n in PLOT_N_EFFS:
        # Z[i, j] = G~(x_j, y_i): rows are y, matching imshow(origin="lower")
        Zdiffs.append(np.abs(G_true - P @ K.T @ P.T))
    print(f"N_eff = {n:3d}: relative Bochner error {rel_boch[-1]:.2e}, relative kernel error {rel_kernel[-1]:.2e}")

np.save(os.path.join(DATA, "ker_N_eff.npy"), N_EFFS)
np.save(os.path.join(DATA, "ker_rel_boch_err.npy"), rel_boch)
np.save(os.path.join(DATA, "ker_rel_kern_err.npy"), rel_kernel)
np.save(os.path.join(DATA, "ker_Zdiffs.npy"), np.array(Zdiffs, dtype=np.float32))

In [ ]:
from matplotlib.colors import LogNorm

fig_width, fig_height = paper_style(width=1.0, aspect=0.9, fontsize=7)

# Errors versus N_eff
fig, ax = plt.subplots(figsize=(0.4 * fig_width, 0.4 * fig_height), layout="constrained")
ax.plot(N_EFFS, rel_boch, "^--", color="k", label="Relative Bochner Norm Error", markersize=4, linewidth=1)
ax.plot(N_EFFS, rel_kernel, "v--", color="red", label="Relative $L^2$ Kernel Error", markersize=4, linewidth=1)
ax.set(xlabel=r"$N_{\mathrm{eff}}$", yscale="log")
ax.legend(loc="best")
savefig(os.path.join(FIGS, "rel_l2_kernel_and_boch_err.pdf"))
plt.show()

# True kernel and pointwise errors
fig, axes = plt.subplots(1, 1 + len(PLOT_N_EFFS), figsize=(0.7 * fig_width, 0.7 * fig_height), layout="constrained")
im0 = axes[0].imshow(G_true, origin="lower", extent=[0, 1, 0, 1], cmap="coolwarm")
fig.colorbar(im0, ax=axes[0], location="left", pad=0.02, fraction=0.05)
axes[0].set_title(r"$G$")
for ax, n, Z in zip(axes[1:], PLOT_N_EFFS, Zdiffs):
    im = ax.imshow(Z, origin="lower", extent=[0, 1, 0, 1], cmap="viridis", norm=LogNorm(1e-7, 1))
    ax.set_title(r"$\vert G - \tilde{G}_{V(%d)}\vert$" % n)
fig.colorbar(im, ax=axes[-1], location="right", pad=0.02, fraction=0.05)
for ax in axes:
    ax.axis("off")
savefig(os.path.join(FIGS, "kernel_pointwise_err.pdf"))
plt.show()